In [1]:
# libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
import warnings
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
warnings.filterwarnings('ignore')

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


>In this notebook we train a pretrained transformer model (DistilBert) and an LSTM model on our sequential data (IMDB dataset), and compare their performance.

## Preprocessing Dataset

>The IMDB dataset is a text based dataset, so we can use it as a sequential data. We want to train our models based on the comments of reviewers and try to predict whether their opinion was positive or negative so it would be a binary classification task.

In [3]:
MAX_FEATURES = 10000  # vocabulary size
MAX_LEN = 200         # sequence length
EMBEDDING_DIM = 128
BATCH_SIZE = 64
EPOCHS = 10

# Load and preprocess IMDB dataset
print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)

# Pad sequences
x_train = sequence.pad_sequences(x_train, maxlen=MAX_LEN)
x_test = sequence.pad_sequences(x_test, maxlen=MAX_LEN)

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Positive samples in train: {sum(y_train)}")
print(f"Negative samples in train: {len(y_train) - sum(y_train)}")

# Get word index for decoding (to convert back to text for transformer)
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(sequence_data):
  
    return ' '.join([reverse_word_index.get(i-3, '?') for i in sequence_data if i > 3])


Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (25000, 200)
Test data shape: (25000, 200)
Training labels shape: (25000,)
Test labels shape: (25000,)
Positive samples in train: 12500
Negative samples in train: 12500
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
# Dataset for RNN/LSTM models (using numeric sequences)"""
class IMDBDataset(Dataset):
 
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# Dataset for Transformer models (converted to text)
class IMDBDatasetTransformer(Dataset):
    
    def __init__(self, data, labels, tokenizer, max_length=200):
        self.data = data
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Convert numeric sequence to text
        text = decode_review(self.data[idx])
        label = self.labels[idx]

        # Tokenize for transformer
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


## LSTM Model

In [15]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=MAX_FEATURES, embedding_dim=EMBEDDING_DIM,
                 hidden_dim=256, num_layers=2, num_classes=2, dropout=0.5, bidirectional=True):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(dropout)

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(lstm_output_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)

        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # For bidirectional LSTM, concatenate last hidden states
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # Classification
        hidden = self.dropout(hidden)
        output = self.fc(hidden)

        return output

# Training
def train_lstm(model, train_loader, val_loader, epochs=EPOCHS, lr=0.001):
 
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        
        model.train()
        total_train_loss = 0
        for batch_idx, (data, labels) in enumerate(train_loader):
            data, labels = data.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_train_loss += loss.item()

            if batch_idx % 50 == 0:
                print(f"LSTM Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        val_accuracy = evaluate_lstm(model, val_loader)
        val_accuracies.append(val_accuracy)

        scheduler.step(avg_train_loss)

        print(f"\nLSTM Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Accuracy: {val_accuracy:.4f}\n")

    return train_losses, val_accuracies

# Evaluation
def evaluate_lstm(model, data_loader):
    """Evaluate LSTM model"""
    model.eval()
    predictions = []
    actual_labels = []

    with torch.no_grad():
        for data, labels in data_loader:
            data = data.to(device)
            outputs = model(data)
            preds = torch.argmax(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            actual_labels.extend(labels.numpy())

    return accuracy_score(actual_labels, predictions)

## Transformer Model

In [12]:
class TransformerModel:
    def __init__(self, num_classes=2, max_length=MAX_LEN):

        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

        self.model = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased',
            num_labels=num_classes
        ).to(device)

        self.max_length = max_length
        self.device = device

    def train(self, train_dataset, val_dataset, epochs=3, batch_size=16):
        """Train Transformer model"""
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Optimizer and scheduler
        optimizer = AdamW(self.model.parameters(), lr=2e-5, weight_decay=0.01)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        train_losses = []
        val_accuracies = []

        for epoch in range(epochs):
            # Training
            self.model.train()
            total_train_loss = 0

            for batch_idx, batch in enumerate(train_loader):
                optimizer.zero_grad()

                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                total_train_loss += loss.item()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

                if batch_idx % 50 == 0:
                    print(f"Transformer Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

            avg_train_loss = total_train_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # Validation
            val_accuracy = self.evaluate(val_loader)
            val_accuracies.append(val_accuracy)

            print(f"\nTransformer Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Accuracy: {val_accuracy:.4f}\n")

        return train_losses, val_accuracies

    def evaluate(self, data_loader):
        """Evaluate Transformer model"""
        self.model.eval()
        predictions = []
        actual_labels = []

        with torch.no_grad():
            for batch in data_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels']

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                preds = torch.argmax(outputs.logits, dim=1)
                predictions.extend(preds.cpu().numpy())
                actual_labels.extend(labels.numpy())

        return accuracy_score(actual_labels, predictions)

    def predict(self, texts):
        """Predict sentiment for new texts"""
        self.model.eval()
        predictions = []

        for text in texts:
            encoding = self.tokenizer(
                text,
                truncation=True,
                padding='max_length',
                max_length=self.max_length,
                return_tensors='pt'
            )

            with torch.no_grad():
                outputs = self.model(
                    input_ids=encoding['input_ids'].to(self.device),
                    attention_mask=encoding['attention_mask'].to(self.device)
                )
                pred = torch.argmax(outputs.logits, dim=1)
                predictions.append(pred.item())

        return predictions

## Train & Compare Models

In [6]:
split_idx = int(0.8 * len(x_train))
x_train_subset = x_train[:split_idx]
y_train_subset = y_train[:split_idx]
x_val_subset = x_train[split_idx:]
y_val_subset = y_train[split_idx:]

print(f"\nTraining samples: {len(x_train_subset)}")
print(f"Validation samples: {len(x_val_subset)}")
print(f"Test samples: {len(x_test)}")


Training samples: 20000
Validation samples: 5000
Test samples: 25000


In [16]:
# Create LSTM data loaders
train_dataset_lstm = IMDBDataset(x_train_subset, y_train_subset)
val_dataset_lstm = IMDBDataset(x_val_subset, y_val_subset)
test_dataset_lstm = IMDBDataset(x_test, y_test)

train_loader_lstm = DataLoader(train_dataset_lstm, batch_size=BATCH_SIZE, shuffle=True)
val_loader_lstm = DataLoader(val_dataset_lstm, batch_size=BATCH_SIZE, shuffle=False)
test_loader_lstm = DataLoader(test_dataset_lstm, batch_size=BATCH_SIZE, shuffle=False)

# Initialize and train LSTM
lstm_model = LSTMClassifier(
    vocab_size=MAX_FEATURES,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=10,
    num_layers=2,
    dropout=0.5,
    bidirectional=True
).to(device)

print(f"LSTM Model parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

lstm_train_losses, lstm_val_acc = train_lstm(
    lstm_model, train_loader_lstm, val_loader_lstm,
    epochs=EPOCHS, lr=0.001
)

# Evaluate LSTM on test set
lstm_test_accuracy = evaluate_lstm(lstm_model, test_loader_lstm)
print(f"\nLSTM Final Test Accuracy: {lstm_test_accuracy:.4f}")

LSTM Model parameters: 1,293,802
LSTM Epoch 1, Batch 0/313, Loss: 0.7006
LSTM Epoch 1, Batch 50/313, Loss: 0.6933
LSTM Epoch 1, Batch 100/313, Loss: 0.6942
LSTM Epoch 1, Batch 150/313, Loss: 0.6815
LSTM Epoch 1, Batch 200/313, Loss: 0.6824
LSTM Epoch 1, Batch 250/313, Loss: 0.6647
LSTM Epoch 1, Batch 300/313, Loss: 0.7117

LSTM Epoch 1/10 - Train Loss: 0.6868, Val Accuracy: 0.6768

LSTM Epoch 2, Batch 0/313, Loss: 0.6858
LSTM Epoch 2, Batch 50/313, Loss: 0.6584
LSTM Epoch 2, Batch 100/313, Loss: 0.6711
LSTM Epoch 2, Batch 150/313, Loss: 0.5996
LSTM Epoch 2, Batch 200/313, Loss: 0.5829
LSTM Epoch 2, Batch 250/313, Loss: 0.5217
LSTM Epoch 2, Batch 300/313, Loss: 0.4971

LSTM Epoch 2/10 - Train Loss: 0.5807, Val Accuracy: 0.7654

LSTM Epoch 3, Batch 0/313, Loss: 0.5256
LSTM Epoch 3, Batch 50/313, Loss: 0.4539
LSTM Epoch 3, Batch 100/313, Loss: 0.5841
LSTM Epoch 3, Batch 150/313, Loss: 0.4376
LSTM Epoch 3, Batch 200/313, Loss: 0.4273
LSTM Epoch 3, Batch 250/313, Loss: 0.4770
LSTM Epoch 3, 

In [13]:
train_subset_size = 5000  # Use 5k for faster training
val_subset_size = 1000

x_train_trans = x_train_subset[:train_subset_size]
y_train_trans = y_train_subset[:train_subset_size]
x_val_trans = x_val_subset[:val_subset_size]
y_val_trans = y_val_subset[:val_subset_size]

print(f"Transformer training samples: {len(x_train_trans)}")
print(f"Transformer validation samples: {len(x_val_trans)}")

# Initialize transformer
transformer = TransformerModel(num_classes=2, max_length=MAX_LEN)

# Create transformer datasets (converts numeric to text internally)
train_dataset_trans = IMDBDatasetTransformer(x_train_trans, y_train_trans, transformer.tokenizer, MAX_LEN)
val_dataset_trans = IMDBDatasetTransformer(x_val_trans, y_val_trans, transformer.tokenizer, MAX_LEN)

# Train transformer
transformer_train_losses, transformer_val_acc = transformer.train(
    train_dataset_trans, val_dataset_trans,
    epochs=3, batch_size=16  # Transformers need fewer epochs
)

# Evaluate transformer on test set (use subset for speed)
test_subset_size = 2000
test_dataset_trans = IMDBDatasetTransformer(
    x_test[:test_subset_size], y_test[:test_subset_size],
    transformer.tokenizer, MAX_LEN
)
test_loader_trans = DataLoader(test_dataset_trans, batch_size=32, shuffle=False)
transformer_test_accuracy = transformer.evaluate(test_loader_trans)
print(f"\nTransformer Final Test Accuracy: {transformer_test_accuracy:.4f}")

Transformer training samples: 5000
Transformer validation samples: 1000


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Transformer Epoch 1, Batch 0/313, Loss: 0.6687
Transformer Epoch 1, Batch 50/313, Loss: 0.6305
Transformer Epoch 1, Batch 100/313, Loss: 0.3250
Transformer Epoch 1, Batch 150/313, Loss: 0.4315
Transformer Epoch 1, Batch 200/313, Loss: 0.1341
Transformer Epoch 1, Batch 250/313, Loss: 0.3279
Transformer Epoch 1, Batch 300/313, Loss: 0.3489

Transformer Epoch 1/3 - Train Loss: 0.4102, Val Accuracy: 0.8540

Transformer Epoch 2, Batch 0/313, Loss: 0.3317
Transformer Epoch 2, Batch 50/313, Loss: 0.2792
Transformer Epoch 2, Batch 100/313, Loss: 0.0610
Transformer Epoch 2, Batch 150/313, Loss: 0.0710
Transformer Epoch 2, Batch 200/313, Loss: 0.2764
Transformer Epoch 2, Batch 250/313, Loss: 0.1801
Transformer Epoch 2, Batch 300/313, Loss: 0.0168

Transformer Epoch 2/3 - Train Loss: 0.2189, Val Accuracy: 0.8860

Transformer Epoch 3, Batch 0/313, Loss: 0.1372
Transformer Epoch 3, Batch 50/313, Loss: 0.2621
Transformer Epoch 3, Batch 100/313, Loss: 0.4723
Transformer Epoch 3, Batch 150/313, Loss: 

>For the LSTM model the test accuracy was 0.85 and for the transformer model it was 0.88, that is slightly higher than the LSTM model.In the next part, we will compare them in more details.

In [17]:
# Get detailed classification reports
print("\nGenerating detailed classification reports...")

# LSTM predictions on test set
lstm_model.eval()
lstm_all_preds = []
lstm_all_labels = []
with torch.no_grad():
    for data, labels in test_loader_lstm:
        data = data.to(device)
        outputs = lstm_model(data)
        preds = torch.argmax(outputs, dim=1)
        lstm_all_preds.extend(preds.cpu().numpy())
        lstm_all_labels.extend(labels.numpy())

print("\n" + "-"*40)
print("LSTM CLASSIFICATION REPORT")
print("-"*40)
print(classification_report(lstm_all_labels, lstm_all_preds,
                            target_names=['Negative', 'Positive']))

# Transformer predictions on test set
transformer.model.eval()
transformer_all_preds = []
transformer_all_labels = []
with torch.no_grad():
    for batch in test_loader_trans:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = transformer.model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        transformer_all_preds.extend(preds.cpu().numpy())
        transformer_all_labels.extend(labels.numpy())

print("\n" + "-"*40)
print("TRANSFORMER CLASSIFICATION REPORT")
print("-"*40)
print(classification_report(transformer_all_labels, transformer_all_preds,
                            target_names=['Negative', 'Positive']))


Generating detailed classification reports...

----------------------------------------
LSTM CLASSIFICATION REPORT
----------------------------------------
              precision    recall  f1-score   support

    Negative       0.89      0.80      0.84     12500
    Positive       0.82      0.90      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.85     25000
weighted avg       0.85      0.85      0.85     25000


----------------------------------------
TRANSFORMER CLASSIFICATION REPORT
----------------------------------------
              precision    recall  f1-score   support

    Negative       0.92      0.86      0.89      1047
    Positive       0.85      0.92      0.89       953

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000



>We trained the transformer model on a subset of the data because it took a long time to train on the whole data. 
As we can see in the report, the transformer model(DistilBert) outperforms the LSTM model using all citeria.

## 1. Main Advantages and Disadvantages of Transformer-Based Models

### Advantages

- **Parallel Processing**: Transformers process all tokens simultaneously, unlike RNNs that must process sequentially, leading to much faster training
- **Long-Range Dependencies**: Self-attention captures relationships between any positions in the sequence regardless of distance
- **Transfer Learning**: Pre-trained models (BERT, GPT, DistilBERT) can be fine-tuned for specific tasks with relatively small amounts of labeled data
- **Contextual Representations**: Creates dynamic word embeddings that change based on surrounding context (unlike static embeddings in traditional models)
- **State-of-the-Art Performance**: Achieves superior results across NLP, computer vision, and audio processing tasks

### Disadvantages

- **Quadratic Complexity**: O(n²) computational complexity in sequence length, making very long sequences extremely expensive
- **Memory Intensive**: Attention matrices require O(n²) memory, limiting maximum sequence length on hardware
- **Data Hungry**: Require large amounts of training data to perform well compared to simpler models
- **Interpretability Challenges**: Attention patterns are complex and difficult to interpret meaningfully
- **Positional Information Loss**: Unlike RNNs with inherent sequential order, transformers need explicit positional encodings
- **Inference Latency**: Can be slower for autoregressive generation (like GPT) compared to RNNs

---

## 2. Why Transformers Scale Well with Data and Model Size

### Scaling with Data

- **No Sequential Bottleneck**: Parallel processing removes the sequential dependency found in RNNs, allowing efficient use of large batches and distributed training
- **Vanishing Gradient Resistance**: Residual connections and layer normalization prevent gradient issues even as networks become very deep
- **Capacity to Learn**: More parameters can capture increasingly complex patterns in massive datasets
- **Empirical Scaling Laws**: Research shows transformer performance improves predictably as a power-law function of dataset size

### Scaling with Model Size

- **Parameter Efficiency**: Self-attention layers have relatively few parameters per layer compared to RNNs/LSTMs
- **Modular Architecture**: Easy to independently scale layers, attention heads, and hidden dimensions
- **Hardware Optimization**: Matrix multiplications (the core of transformers) are highly optimized on modern GPUs and TPUs
- **Sparse Attention Variants**: Advanced architectures (sparse, linear, sliding window attention) reduce complexity for further scaling

---

## 3. Why Transformers Require Large Computational Resources

### Primary Reasons

**Quadratic Complexity**
- Self-attention computes pairwise interactions between every pair of tokens
- A sequence of length 512 requires over 262,000 attention scores
- A sequence of length 2048 requires over 4 million attention scores

**Memory Footprint**
- Must store the entire attention matrix for each layer and each head
- For batch size 32, sequence length 512: approximately 134 million values in memory
- 12 attention heads multiply this by 12

**Layer Stacking**
- Deep networks (BERT-large has 24 layers) multiply computational and memory requirements
- Each layer performs full attention and feed-forward computations

**Comparison with LSTM**

| Aspect | LSTM | Transformer |
|--------|------|-------------|
| Sequential processing | Required | Not required |
| Path length for long-range dependencies | O(n) steps | O(1) direct connection |
| Memory for attention matrix | None | O(n²) per layer |
| Training parallelization | Limited | Full |
| Parameter count (comparable model) | ~28M | ~66M (DistilBERT) |

---

# 4. What is Self-Attention and What Problem Does It Solve?

### The Problem

RNNs and LSTMs have a fundamental **information bottleneck**: data must flow sequentially through each timestep. For long sentences, early words lose connection to later words due to:

- **Vanishing gradients**: Information decays exponentially along long paths
- **Limited memory**: Hidden states have finite capacity to retain information
- **Sequential processing**: Cannot directly connect distant positions

### The Self-Attention Solution

Self-attention computes **direct relationships** between all positions in a sequence simultaneously. It works through three concepts:

- **Query (Q)**: Represents what each position is "looking for"
- **Key (K)**: Represents what features each position has
- **Value (V)**: Represents the actual information each position carries

The attention score between any two positions is computed by comparing Query of one with Keys of all others, then using these scores to create a weighted sum of Values.

### What Problem Does It Solve?

- **Long-Range Dependencies**: Creates direct connections between any positions regardless of distance
- **Parallel Processing**: All relationships computed simultaneously, not sequentially
- **Dynamic Context**: Attention weights change based on input content (unlike fixed filters in CNNs)
- **Vanishing Gradients**: Gradient paths are direct, eliminating sequential decay

---
## 5. Why Self-Attention Models Long-Range Dependencies Better Than RNNs

### The RNN Limitation

In an RNN or LSTM, information flows through a chain:

- **Path length** between first and last token: O(n) steps
- Each step risks information loss or corruption
- Maximum effective distance: typically 50-100 tokens even for LSTMs

### The Transformer Advantage

With self-attention:

- **Path length** between any two tokens: **1 step** (direct connection)
- No sequential degradation regardless of distance
- Position 100 can directly attend to position 1

### Mathematical Comparison

- **RNN**: Hidden state h_t = f(h_{t-1}, x_t) — information flows sequentially through h₁ → h₂ → ... → h₁₀₀
- **Transformer**: Attention(q_i, k_j) for all i,j — position 100 can directly query position 1

### Gradient Flow

| Model | Path Length | Vanishing Gradient Risk |
|-------|-------------|------------------------|
| Simple RNN | O(n) | High (exponential decay) |
| LSTM | O(n) | Medium (gated decay) |
| Transformer | O(1) | Low (direct skip connections) |

### Example: Long Dependency

Consider: *"The cat that lived in the house on the hill with the red door finally slept"*

- **RNN**: "slept" (word 20) connects to "cat" (word 2) through 18 sequential steps
- **Transformer**: "slept" directly attends to "cat" in one attention computation

---
## 6. What is Multi-Head Attention and Why Does It Help?

### The Limitation of Single-Head Attention

One attention head learns only **one type of relationship** between tokens. For example, it might learn syntactic patterns (subject-verb agreement) but miss semantic relationships (product-category associations).

### Multi-Head Attention Solution

Multi-head attention runs **multiple attention mechanisms in parallel**, each with its own learned projections. The outputs are concatenated and projected to form the final representation.

### Why Multiple Heads Help

- **Different Perspectives**: Each head learns to focus on different types of relationships
- **Richer Representations**: Concatenated outputs capture more complex, multi-faceted patterns
- **Ensemble Effect**: Multiple "specialists" working together, each contributing unique insights
- **Improved Interpretability**: Different heads often correspond to interpretable linguistic phenomena

### Example of Different Heads

In a typical transformer, different heads might specialize in:

- **Head 1**: Syntactic dependencies (subject-verb, noun-adjective)
- **Head 2**: Semantic relationships (product-brand, person-occupation)
- **Head 3**: Coreference resolution (pronoun → antecedent)
- **Head 4**: Negation scope and sentiment patterns
- **Head 5**: Temporal relationships (before/after)
- **Head 6**: Discourse markers and rhetorical structure

### Practical Impact

- **Single head**: "excellent" connects strongly to "movie" (sentiment)
- **Multiple heads**: "excellent" also connects to "acting" (aspect-based), "highly" (intensity), and "recommend" (action)

---
## 7. What is the Role of Positional Encoding?

### The Fundamental Problem

Self-attention is **permutation-invariant** — it treats sequences as unordered sets. Without positional information, the model cannot distinguish:

- *"I love cats"* from *"cats love I"*
- *"John hit Mary"* from *"Mary hit John"*

Both would produce identical attention outputs despite having completely different meanings.

### The Solution: Positional Encoding

Positional encoding adds **position information** to token embeddings:

- **Final Embedding** = Word Embedding + Positional Encoding
- Each position receives a unique vector encoding its location
- The model can now distinguish word order

### Sinusoidal Positional Encoding

The original transformer used fixed sinusoidal functions:

- **Even dimensions**: Using sine function
- **Odd dimensions**: Using cosine function
- Each position gets a unique encoding pattern
- Values are bounded between -1 and 1, allowing addition to word embeddings

### Why Sinusoidal Encodings Work Well

- **Unique per Position**: Every position has a distinct encoding
- **Bounded Values**: Stays within [-1, 1] range, stable for addition
- **Relative Position Property**: The encoding for position (pos+k) can be expressed as a linear function of the encoding for position pos
- **No Learned Parameters**: Works for sequences longer than seen during training

### Alternative Approaches

| Method | Description | Trade-off |
|--------|-------------|-----------|
| **Sinusoidal (fixed)** | Mathematical functions | No parameters, unlimited length, less flexible |
| **Learned** | Embedding lookup table | Flexible, task-specific, limited to max length |
| **Relative** | Encode distances between positions | Better for very long sequences, more complex |

### Without vs. With Positional Encoding

**Without positional encoding:**
- Inputs [I, love, cats] and [cats, love, I] produce same representation
- Word order information is completely lost
- Model cannot learn syntax or sequential patterns

**With positional encoding:**
- Position 0=I, position 1=love, position 2=cats → unique representation
- Position 0=cats, position 1=love, position 2=I → different representation
- Model can learn that word order matters


